# Template ground truth manual → validasi OCR

Notebook ini membantu Anda **mengisi teks yang benar secara manual** pada sebagian gambar test, lalu mengukur seberapa baik hasil YOLO + EasyOCR terhadap label tersebut.

**Alur**
1. Jalankan sel 1–3: membuat `ground_truth.csv` berisi sampel acak id test (tidak menimpa file yang sudah ada).
2. Sel 4: antarmuka pelabelan. Gambar crop `entete` ditampilkan, Anda mengetik teksnya. Tiap klik langsung tersimpan, sehingga bisa dihentikan dan dilanjutkan kapan saja.
3. Sel 5–6: skor (exact match dan CER) serta uji beberapa nilai `MIN_CONF` untuk memilih yang terbaik.

**Catatan**
- Secara default prediksi model **disembunyikan** (`SHOW_PREDICTION = False`) agar label tidak bias oleh tebakan model. Set `True` bila ingin lebih cepat, dengan risiko skor jadi terlalu optimis.
- Ketikan otomatis dinormalisasi ke format lomba (huruf besar; hanya `A-Z 0-9 / -`; tanpa spasi).
- Alternatif tanpa widget: edit `ground_truth.csv` di editor teks atau Excel dan isi kolom `true_text`. Di Excel, ubah format kolom `true_text` menjadi **Text** terlebih dulu, karena teks seperti `45/12` akan diubah Excel menjadi tanggal. Isi `status` dengan `tidak_terbaca` bila gambarnya memang tidak terbaca (baris itu tidak ikut dinilai).
- Dengan puluhan label saja, skor punya ketidakpastian beberapa persen. Gunakan untuk membandingkan pengaturan, bukan sebagai angka pasti.

In [31]:
# 1) Konfigurasi
import json, os, random, re
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

TRAIN_CSV = 'train.csv'
TRAIN_DIR = 'train'
TEST_CSV  = 'test.csv'
OCR_CACHE = 'ocr_yolo.json'     # cache OCR dari notebook utama (atau 'ocr_full.json')
CROP_DIR  = 'test_crops'        # crop entete hasil notebook utama
TEST_DIR  = 'test'              # gambar asli (dipakai bila crop tidak ada)
GT_CSV    = 'ground_truth.csv'

N_SAMPLE  = 60                  # jumlah gambar yang akan dilabeli
SEED      = 42
MIN_CONF  = 0.15                # sama dengan pengaturan di sel submission
SHOW_PREDICTION = False         # True = tampilkan & isi otomatis tebakan model (lebih cepat, tapi bias)

In [32]:
# 2) Fungsi bantu (definisi saja): pembersih teks, penyusun teks, dan metrik
_DASH = str.maketrans({'–': '-', '—': '-', '‒': '-', '−': '-', '_': '-', '\\': '/'})
_VALID = re.compile(r'[^A-Z0-9/-]')
_FORMAT = re.compile(r'[A-Z0-9/-]+')


def clean_text(s):
    """Huruf besar; hanya A-Z, 0-9, '/' dan '-'; semua spasi/baris baru hilang."""
    return _VALID.sub('', s.translate(_DASH).upper())


def to_items(dets):
    items = []
    for d in dets:
        xs = [p[0] for p in d['box']]
        ys = [p[1] for p in d['box']]
        x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
        w, h = x2 - x1, y2 - y1
        txt = clean_text(d['text'])
        items.append(dict(x1=x1, x2=x2, w=w, h=h, cy=(y1 + y2) / 2, text=txt, conf=d['conf'],
                          vertical=(h > 1.5 * w and len(txt) >= 2)))
    return [it for it in items if it['text']]


def merge_lines(items, y_tol=0.6, gap_tol=2.5):
    """Gabungkan potongan teks yang sebaris (EasyOCR sering memecah satu ID menjadi beberapa bagian)."""
    mean = lambda v: sum(v) / len(v)
    lines = []
    for it in sorted([i for i in items if not i['vertical']], key=lambda i: i['cy']):
        for ln in lines:
            cy = mean([m['cy'] for m in ln])
            hh = max(it['h'], mean([m['h'] for m in ln]))
            gap = min(max(0, it['x1'] - m['x2'], m['x1'] - it['x2']) for m in ln)
            if abs(it['cy'] - cy) < y_tol * hh and gap < gap_tol * hh:
                ln.append(it)
                break
        else:
            lines.append([it])
    out = []
    for ln in lines:
        ln = sorted(ln, key=lambda m: m['x1'])
        wts = [len(m['text']) for m in ln]
        out.append(dict(text=''.join(m['text'] for m in ln), cy=mean([m['cy'] for m in ln]),
                        conf=sum(m['conf'] * w for m, w in zip(ln, wts)) / sum(wts)))
    out += [dict(text=i['text'], cy=i['cy'], conf=i['conf']) for i in items if i['vertical']]
    return out


def crop_text(dets, min_conf):
    """Isi crop `entete`: semua baris dibaca atas -> bawah lalu digabung tanpa spasi."""
    lines = merge_lines([it for it in to_items(dets) if it['conf'] >= min_conf])
    lines.sort(key=lambda l: l['cy'])
    return ''.join(l['text'] for l in lines).strip('/-')


def score_line(ln):
    t = ln['text']
    if len(t.strip('/-')) < 3:
        return 0.0
    s = ln['conf'] * min(len(t), 20)
    if any(c.isdigit() for c in t) and any(c in '/-' for c in t):   # pola khas ID: angka + pemisah
        s *= 1.5
    return s


def best_line_text(dets):
    """Cadangan / mode full: pilih SATU baris terbaik dari seluruh gambar."""
    lines = merge_lines(to_items(dets))
    best = max(lines, key=score_line, default=None)
    if best is not None and score_line(best) > 0:
        return best['text'].strip('/-') or '0'
    return '0'


def final_text(entry, min_conf):
    if entry.get('src') == 'crop':
        return crop_text(entry['dets'], min_conf) or '0'
    return best_line_text(entry.get('dets', []))


def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def cer(true, pred):
    """Character error rate = jarak edit / panjang teks benar."""
    return edit_distance(true, pred) / max(1, len(true))

In [33]:
# 3) Buat template ground_truth.csv (atau muat yang sudah ada)
with open(OCR_CACHE, encoding='utf-8') as f:
    cache = json.load(f)

def load_gt():
    return pd.read_csv(GT_CSV, dtype=str, keep_default_na=False)

def save_gt(df):
    df.to_csv(GT_CSV, index=False)

def find_image(img_id):
    stem = os.path.splitext(img_id)[0]
    for p in (os.path.join(CROP_DIR, stem + '.png')):
        if os.path.exists(p):
            return p
    return ''

pick = []
for i in range(50):
    pick.append(i)

if os.path.exists(GT_CSV):
    gt = load_gt()
    print(f"Memuat {GT_CSV} yang sudah ada ({len(gt)} baris); tidak ditimpa.")
else:
    ids = pd.read_csv(CROP_DIR, dtype=str).iloc[:, 0].tolist()
    pool = [i for i in ids if i in cache]
     # hanya gambar yang sudah di-OCR
    gt = pd.DataFrame({'id': pick, 'image_path': [find_image(i) for i in pick],
                       'true_text': '', 'status': ''})
    save_gt(gt)
    print(f"Template dibuat: {GT_CSV} ({len(gt)} baris dari {len(pool)} gambar ber-OCR)")

print("Kolom: id | image_path | true_text (isi teks benar) | status ('' / 'ok' / 'tidak_terbaca')")
gt.head()

Memuat ground_truth.csv yang sudah ada (60 baris); tidak ditimpa.
Kolom: id | image_path | true_text (isi teks benar) | status ('' / 'ok' / 'tidak_terbaca')


,id,image_path,true_text,status
0,f817cf172cea52c2333029c84177c65f.jpg,test_crops\f817cf172cea52c2333029c84177c65f.png,,
1,287bbd5943c5b48970574a5f7e5a5270.jpg,test\287bbd5943c5b48970574a5f7e5a5270.jpg,,
2,07fccecb1c1525329ad094c7104bce8c.jpg,test_crops\07fccecb1c1525329ad094c7104bce8c.png,,
3,6d5d47083586849ca16e8f05c47ec1ff.jpg,test\6d5d47083586849ca16e8f05c47ec1ff.jpg,,
4,610011164f25ff910162877de254e736.jpg,test_crops\610011164f25ff910162877de254e736.png,,


In [34]:
# 4) Antarmuka pelabelan (tersimpan otomatis tiap klik; aman dihentikan & dilanjutkan)
try:
    import ipywidgets as W
except ImportError:
    W = None

def pred_of(img_id):
    return final_text(cache[img_id], MIN_CONF) if img_id in cache else '0'

if W is None:
    print("ipywidgets belum terpasang (pip install ipywidgets). Alternatif: isi kolom true_text di "
          f"{GT_CSV} secara manual, lalu lanjut ke sel 5.")
else:
    gt = load_gt()
    todo = gt.index[(gt.true_text == '') & (gt.status == '')].tolist()
    S = dict(i=todo[0] if todo else 0)

    head = W.HTML(); hint = W.HTML(); msg = W.HTML()
    img = W.Image(width=720)
    txt = W.Text(placeholder='ketik teks pada stiker lalu Enter', layout=W.Layout(width='520px'))
    b_prev = W.Button(description='Sebelumnya')
    b_bad  = W.Button(description='Tidak terbaca')
    b_skip = W.Button(description='Lewati')
    b_save = W.Button(description='Simpan & berikut', button_style='success')

    def show(i):
        S['i'] = i
        if i >= len(gt):
            img.value = b''
            head.value = '<b>Selesai</b>: semua baris sudah dilewati/diisi. Lanjut ke sel 5.'
            hint.value = ''; txt.value = ''
            return
        row = gt.iloc[i]
        if row.image_path and os.path.exists(row.image_path):
            img.format = 'png' if row.image_path.lower().endswith('.png') else 'jpeg'
            with open(row.image_path, 'rb') as f:
                img.value = f.read()
        else:
            img.value = b''
        done = int(((gt.true_text != '') | (gt.status == 'tidak_terbaca')).sum())
        head.value = f"<b>{i + 1} / {len(gt)}</b> &nbsp; (terisi: {done}) &nbsp; <code>{row.id}</code>"
        hint.value = f"Prediksi model: <code>{pred_of(row.id)}</code>" if SHOW_PREDICTION else ''
        txt.value = row.true_text or (pred_of(row.id) if SHOW_PREDICTION else '')

    def commit(status, text):
        i = S['i']
        if i >= len(gt):
            return
        gt.loc[gt.index[i], ['true_text', 'status']] = [text, status]
        save_gt(gt)
        show(i + 1)

    def on_save(_):
        text = clean_text(txt.value)
        if not text:
            msg.value = 'Isi teks dulu, atau pilih "Tidak terbaca" / "Lewati".'
            return
        commit('ok', text)
        msg.value = f'Tersimpan: <code>{text}</code>'

    def on_bad(_):
        commit('tidak_terbaca', '')
        msg.value = 'Ditandai tidak terbaca (tidak ikut dinilai).'

    b_save.on_click(on_save)
    b_bad.on_click(on_bad)
    b_skip.on_click(lambda _: show(min(S['i'] + 1, len(gt))))
    b_prev.on_click(lambda _: show(max(0, S['i'] - 1)))
    try:
        txt.on_submit(on_save)                       # tekan Enter = simpan
    except Exception:
        pass

    show(S['i'])
    display(W.VBox([head, img, hint, txt, W.HBox([b_prev, b_bad, b_skip, b_save]), msg]))

ipywidgets belum terpasang (pip install ipywidgets). Alternatif: isi kolom true_text di ground_truth.csv secara manual, lalu lanjut ke sel 5.


In [35]:
# 5) Skor terhadap ground truth (membaca ulang ground_truth.csv, jadi edit manual ikut terbaca)
gt = load_gt()
labeled = gt[(gt.true_text != '') & (gt.status != 'tidak_terbaca')].copy()
labeled['true_text'] = labeled['true_text'].map(clean_text)
labeled = labeled[labeled.true_text != '']

print(f"Total baris: {len(gt)} | berlabel: {len(labeled)} | tidak terbaca: {int((gt.status == 'tidak_terbaca').sum())} "
      f"| belum diisi: {int(((gt.true_text == '') & (gt.status == '')).sum())}")

if labeled.empty:
    print("Belum ada label. Isi dulu lewat sel 4 atau langsung di ground_truth.csv.")
else:
    labeled['pred'] = [pred_of(i) for i in labeled.id]
    labeled['src'] = [cache.get(i, {}).get('src', 'belum_ocr') for i in labeled.id]
    labeled['exact'] = labeled.true_text == labeled.pred
    labeled['edit'] = [edit_distance(t, p) for t, p in zip(labeled.true_text, labeled.pred)]
    labeled['cer'] = [cer(t, p) for t, p in zip(labeled.true_text, labeled.pred)]

    print(f"\nExact match : {labeled.exact.mean():.1%}  ({int(labeled.exact.sum())}/{len(labeled)})")
    print(f"CER rata-rata: {labeled.cer.mean():.3f}  (0 = sempurna)")
    print(f"Median jarak edit: {labeled.edit.median():.1f} karakter")
    print("\nPer sumber teks:")
    display(labeled.groupby('src').agg(n=('exact', 'size'), exact=('exact', 'mean'), cer=('cer', 'mean')).round(3))
    print("Kesalahan (diurutkan dari yang terburuk):")
    display(labeled[~labeled.exact].sort_values('cer', ascending=False)[['id', 'true_text', 'pred', 'src', 'edit', 'cer']])

Total baris: 60 | berlabel: 0 | tidak terbaca: 0 | belum diisi: 60
Belum ada label. Isi dulu lewat sel 4 atau langsung di ground_truth.csv.


In [36]:
# 6) Uji beberapa nilai MIN_CONF pada label yang ada (tanpa OCR ulang)
if labeled.empty:
    print("Belum ada label.")
else:
    rows = []
    for mc in [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]:
        preds = [final_text(cache[i], mc) if i in cache else '0' for i in labeled.id]
        rows.append(dict(MIN_CONF=mc,
                         exact=sum(p == t for p, t in zip(preds, labeled.true_text)) / len(labeled),
                         CER=sum(cer(t, p) for t, p in zip(labeled.true_text, preds)) / len(labeled)))
    sweep = pd.DataFrame(rows).round(3)
    display(sweep)
    best = sweep.sort_values(['exact', 'CER'], ascending=[False, True]).iloc[0]
    print(f"Terbaik pada sampel ini: MIN_CONF = {best.MIN_CONF} (exact {best.exact:.1%}, CER {best.CER:.3f})")
    print("Terapkan di sel submission notebook utama (atau make_submission) bila selisihnya cukup berarti.")

Belum ada label.
